In [0]:
dbutils.widgets.text('source_db','cleansed')
dbutils.widgets.text('mart_db','mart')
dbutils.widgets.text('table_name','cancellation')
dbutils.widgets.text('scd_type','SCD1')
dbutils.widgets.text('primary_key','Code')
dbutils.widgets.text('partition_key','Code')

In [0]:
def create_dim_table(catalog,source_db,mart_db,table_name,scd_type,primary_key,partition_key):
    try:
        # spark.conf.set("spark.databricks.delta.schema.autoMerge.enabled", "true")
        from delta.tables import DeltaTable
        from pyspark.sql.functions import col, current_date, lit,current_user

        source_db = dbutils.widgets.get('source_db')
        mart_db = dbutils.widgets.get('mart_db')
        table_name = dbutils.widgets.get('table_name')
        scd_type = dbutils.widgets.get('scd_type')
        primary_key = dbutils.widgets.get('primary_key')
        partition_key = dbutils.widgets.get('partition_key')

        source_table = f"projectdev.{source_db}.{table_name}"
        df_source = spark.table(source_table)

        df_source = df_source.dropDuplicates()

        # Optionally, filter by latest updates (incremental)
        # df_source = df_source.filter(col('updated_at') > col('last_load_ts'))

        if scd_type.upper() == "SCD1":
            mart_table = f"{catalog}.{mart_db}.dim_{table_name}"
            # Create mart table if not exists
            if not spark.catalog.tableExists(mart_table):
                df_init = df_source.withColumn("created_at", current_date()) \
                                .withColumn("created_by", current_user())\
                                .withColumn("updated_at", current_date()) \
                                .withColumn("updated_by", current_user())

                df_init.write.format("delta") \
                            .mode("overwrite") \
                            .option('mergeSchema','true')\
                            .partitionBy(partition_key)\
                            .saveAsTable(mart_table)
                print(f"✅ Initial load completed: {mart_table}")
            else:
                tgt_df = DeltaTable.forName(spark, mart_table)
                # aligned_df = align_schema(df_source, mart_table)
                # Null-safe change detection
                change_condition = " OR ".join([f"NOT (tgt.{c} <=> src.{c})" for c in df_source.columns if c != primary_key])
                merge_condition = f"tgt.{primary_key} = src.{primary_key}"
                tgt_df.alias("tgt")\
                .merge(df_source.alias("src"), merge_condition)\
                .whenMatchedUpdate(
                    condition=change_condition,
                    set={
                        **{c: col(f"src.{c}") for c in df_source.columns},
                        "updated_at": current_date(),
                        "updated_by": current_user()
                    }
                )\
                .whenNotMatchedInsert(
                    values={
                        **{c: col(f"src.{c}") for c in df_source.columns},
                        "created_at": current_date(),
                        "created_by": current_user(),
                        "updated_at": current_date(),
                        "updated_by": current_user()
                    }
                ).execute()
                print(f"✅ Incremental SCD1 merge completed: {mart_table}")
        elif scd_type.upper() == "SCD2":
            mart_table = f"{catalog}.{mart_db}.dim_{table_name}_scd2"
            df_with_audit = df_source \
            .withColumn("created_at", current_date()) \
            .withColumn("updated_at", current_date()) \
            .withColumn("created_by", current_user()) \
            .withColumn("updated_by", current_user())
            
            if not spark.catalog.tableExists(mart_table):
                df_scd2 = df_with_audit.withColumn("startDate", current_date()) \
                        .withColumn("endDate", lit(None).cast("date")) \
                        .withColumn("IsActive", lit(True))
                df_scd2.write\
                        .format("delta") \
                        .mode("overwrite") \
                        .option('mergeSchema','true')\
                        .partitionBy(partition_key)\
                        .saveAsTable(mart_table)
                        
                print(f"SCD2: {table_name} mart table created with history tracking")
            else:
                delta_table = DeltaTable.forName(spark, mart_table)
                
                merge_condition = f"target.{primary_key} = source.{primary_key} AND target.IsActive = TRUE"
                update_condition = " OR ".join([f"target.{c} <> source.{c}" for c in df_source.columns if c != primary_key])
                
                delta_table.alias("target").merge(
                    df_source.alias("source"),
                    merge_condition
                ).whenMatchedUpdate(
                    condition=update_condition,
                    set={
                        "endDate": current_date(),
                        "IsActive": lit(False),
                        "updated_at": current_date(),
                        "updated_by": current_user()
                    }
                ).whenNotMatchedInsert(
                    values={c: col(c) for c in df_source.columns} | 
                        {"startDate": current_date(),
                        "endDate": lit(None).cast("date"),
                        "IsActive": lit(True),
                        "created_at": current_date(),
                        "updated_at": current_date(),
                        "created_by": current_user(),
                        "updated_by": current_user()}
                ).execute()
                
                print(f"SCD2: {table_name} mart table updated with history")
    except Exception as e:
        import traceback
        print(traceback.format_exc())
        raise e


In [0]:
# %sql
# SET spark.databricks.delta.optimizeWrite = true; 
# SET spark.databricks.delta.autoCompact = true;

In [0]:
%sql
select * from mart.dim_cancellation limit 1

In [0]:
from delta.tables import DeltaTable
from pyspark.sql.functions import col, lit, current_timestamp, current_user, max as spark_max


def create_dim_table(
    catalog,
    source_db,
    mart_db,
    metadata_db,
    table_name,
    scd_type,
    primary_key,
    partition_key=None
):
    source_table = f"{catalog}.{source_db}.{table_name}"
    mart_table = f"{catalog}.{mart_db}.dim_{table_name}"
    watermark_table = f"{catalog}.{metadata_db}.watermark_control"

    watermark_query = f"""
        SELECT COALESCE(MAX(last_watermark), TIMESTAMP('2000-01-01'))
        AS last_watermark
        FROM {watermark_table}
        WHERE table_name = '{table_name}'
    """
    last_watermark = spark.sql(watermark_query).collect()[0]["last_watermark"]

    df_source = (
        spark.table(source_table)
        .filter(col("updated_at") > lit(last_watermark))
        .dropDuplicates([primary_key])
    )

    if df_source.rdd.isEmpty():
        print(f"✅ No new records for {table_name}")
        return

    df_source = (
        df_source
        .withColumn("updated_at", current_timestamp())
        .withColumn("updated_by", current_user())
    )

    if not spark.catalog.tableExists(mart_table):
        df_init = (
            df_source
            .withColumn("created_at", current_timestamp())
            .withColumn("created_by", current_user())
        )

        writer = (
            df_init.write
            .format("delta")
            .mode("overwrite")
            .option("mergeSchema", "true")
        )

        if partition_key:
            writer = writer.partitionBy(partition_key)

        writer.saveAsTable(mart_table)

        print(f"✅ Initial mart load complete: {mart_table}")

    else:
        delta_table = DeltaTable.forName(spark, mart_table)

        if scd_type.upper() == "SCD1":
            business_cols = [
                c for c in df_source.columns
                if c not in ["created_at", "created_by", "updated_at", "updated_by"]
            ]

            merge_condition = f"tgt.{primary_key} = src.{primary_key}"

            delta_table.alias("tgt").merge(
                df_source.alias("src"),
                merge_condition
            ).whenMatchedUpdate(
                set={
                    **{c: col(f"src.{c}") for c in business_cols},
                    "updated_at": current_timestamp(),
                    "updated_by": current_user()
                }
            ).whenNotMatchedInsert(
                values={
                    **{c: col(f"src.{c}") for c in business_cols},
                    "created_at": current_timestamp(),
                    "created_by": current_user(),
                    "updated_at": current_timestamp(),
                    "updated_by": current_user()
                }
            ).execute()

            print(f"✅ SCD1 incremental merge complete: {mart_table}")

        elif scd_type.upper() == "SCD2":
            merge_condition = f"""
                tgt.{primary_key} = src.{primary_key}
                AND tgt.IsActive = TRUE
            """

            delta_table.alias("tgt").merge(
                df_source.alias("src"),
                merge_condition
            ).whenMatchedUpdate(
                condition=f"tgt.updated_at <> src.updated_at",
                set={
                    "endDate": current_timestamp(),
                    "IsActive": lit(False),
                    "updated_at": current_timestamp(),
                    "updated_by": current_user()
                }
            ).whenNotMatchedInsert(
                values={
                    **{c: col(f"src.{c}") for c in df_source.columns},
                    "startDate": current_timestamp(),
                    "endDate": lit(None),
                    "IsActive": lit(True),
                    "created_at": current_timestamp(),
                    "created_by": current_user(),
                    "updated_at": current_timestamp(),
                    "updated_by": current_user()
                }
            ).execute()

            print(f"✅ SCD2 incremental merge complete: {mart_table}")

    new_watermark = df_source.agg(spark_max("updated_at")).collect()[0][0]

    spark.sql(f"""
        MERGE INTO {watermark_table} tgt
        USING (
            SELECT '{table_name}' AS table_name,
                   TIMESTAMP('{new_watermark}') AS last_watermark
        ) src
        ON tgt.table_name = src.table_name
        WHEN MATCHED THEN
            UPDATE SET tgt.last_watermark = src.last_watermark
        WHEN NOT MATCHED THEN
            INSERT *
    """)
    
    spark.sql(f"OPTIMIZE {mart_table}")